<a href="https://colab.research.google.com/github/tusharma006/PYTHON_AI_PROJECTS/blob/main/songs_recomendation_sys/recomendationV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
langchain-chroma \
sentence-transformers \
chromadb \
gradio
!pip install -q langchain-text-splitters

In [23]:
import json
import os
from google.colab import files
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import gradio as gr
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
# Google Colab runs on a remote machine, so we need to
# upload our dataset before we can read it.
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print("\nUploaded Files:")
for file_name in uploaded.keys():
    print(file_name)

Saving songs.json to songs (5).json

Uploaded Files:
songs (5).json


In [5]:
#how to use json and read ?
# import json

# # Open the JSON file
# with open("songs.json", "r", encoding="utf-8") as f:

#     # Convert JSON into Python objects
#     songs = json.load(f)

# # Display information
# print(type(songs))          # list
# print(type(songs[0]))       # dict
# print(songs[0])             # first song
# print(songs[0]["artist"])   # artist name

In [ ]:
# # ==========================================================
# # APPROACH 2 : MULTIPLE JSON FILES
# #
# # Use this approach when you have multiple datasets.
# #
# # Example:
# # songs1.json
# # songs2.json
# # songs3.json
# #
# # All songs will be merged into one Python list.
# # ==========================================================

# import json
# from google.colab import files

# # Upload multiple JSON files
# uploaded = files.upload()

# # Empty list to store songs from every file
# all_songs = []

# # Read every uploaded file
# for file_name in uploaded.keys():

#     # Open the current JSON file
#     with open(file_name, "r", encoding="utf-8") as file:

#         # Convert JSON into Python objects
#         songs = json.load(file)

#     # Add all songs into one list
#     all_songs.extend(songs)

# # Rename for consistency
# songs = all_songs

# # Verify the merged dataset
# print("Datasets Merged Successfully")
# print("Total Songs :", len(songs))
# print("Data Type :", type(songs))
# # ==========================================================
# # APPROACH 3 : MERGE MULTIPLE JSON FILES
# #
# # Upload multiple JSON files
# # Merge them
# # Save them as one dataset
# # Load the merged dataset
# #
# # This is useful if you want to reuse the merged
# # dataset later.
# # ==========================================================

# import json
# from google.colab import files

# # Upload multiple JSON files
# uploaded = files.upload()

# # Store songs from every file
# all_songs = []

# # Read every uploaded JSON file
# for file_name in uploaded.keys():

#     with open(file_name, "r", encoding="utf-8") as file:
#         songs = json.load(file)

#     # Merge the songs
#     all_songs.extend(songs)

# # ----------------------------------------------------------
# # Save merged dataset
# # ----------------------------------------------------------

# merged_file = "merged_songs.json"

# with open(merged_file, "w", encoding="utf-8") as file:
#     json.dump(
#         all_songs,
#         file,
#         indent=4,
#         ensure_ascii=False
#     )

# print("Merged Dataset Saved Successfully")

# # ----------------------------------------------------------
# # Read merged dataset
# # ----------------------------------------------------------

# with open(merged_file, "r", encoding="utf-8") as file:
#     songs = json.load(file)

# # Verify
# print("Merged Dataset Loaded Successfully")
# print("Total Songs :", len(songs))
# print("Data Type :", type(songs))

In [16]:
import json

# Open the uploaded JSON file
with open(file_name, "r", encoding="utf-8") as file:
    songs = json.load(file)
# Verify the dataset
print(file_name)
print("Dataset loaded successfully.\n")
print("Type of songs:", type(songs))
print("Total Songs:", len(songs))
print("\nFirst Song:\n")
print(songs[0])

songs (5).json
Dataset loaded successfully.

Type of songs: <class 'list'>
Total Songs: 50

First Song:

{'id': 1, 'song_name': 'Believer', 'artist': 'Imagine Dragons', 'genre': 'Rock', 'mood': 'Energetic', 'tempo': 'Fast', 'language': 'English', 'release_year': 2017, 'keywords': ['motivation', 'power', 'workout', 'energy'], 'description': 'A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.'}


In [19]:
documents = []
for song in songs:

    page_content = f"""
Song Name : {song["song_name"]}
Artist : {song["artist"]}
Genre : {song["genre"]}
Mood : {song["mood"]}
Language : {song["language"]}
Release Year : {song["release_year"]}
Keywords : {", ".join(song["keywords"])}
Description : {song["description"]}
"""
    document = Document(
        page_content=page_content,
        metadata=song.copy()
    )
    documents.append(document)
print("Total Documents :", len(documents))
print("\nFirst Document\n")
print(documents[0])

Total Documents : 50

First Document

page_content='
Song Name : Believer
Artist : Imagine Dragons
Genre : Rock
Mood : Energetic
Language : English
Release Year : 2017
Keywords : motivation, power, workout, energy
Description : A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.
' metadata={'id': 1, 'song_name': 'Believer', 'artist': 'Imagine Dragons', 'genre': 'Rock', 'mood': 'Energetic', 'tempo': 'Fast', 'language': 'English', 'release_year': 2017, 'keywords': ['motivation', 'power', 'workout', 'energy'], 'description': 'A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.'}


In [22]:
for document in documents:
    print(document.page_content)
    print(document.metadata)


Song Name : Believer
Artist : Imagine Dragons
Genre : Rock
Mood : Energetic
Language : English
Release Year : 2017
Keywords : motivation, power, workout, energy
Description : A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.

{'id': 1, 'song_name': 'Believer', 'artist': 'Imagine Dragons', 'genre': 'Rock', 'mood': 'Energetic', 'tempo': 'Fast', 'language': 'English', 'release_year': 2017, 'keywords': ['motivation', 'power', 'workout', 'energy'], 'description': 'A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.'}

Song Name : Thunder
Artist : Imagine Dragons
Genre : Rock
Mood : Energetic
Language : English
Release Year : 2017
Keywords : success, dreams, energy, rock
Description : An upbeat rock song about chasing dreams with electronic influences and an energetic rhythm.

{'id': 2, 'song_name': 'Thunder', 'artist': 'Imagine Dragons', 'g

In [26]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    # Characters shared between consecutive chunks
    chunk_overlap=50,
    # # Split using these separators
    # separators=["\n\n", "\n", " ", ""]
)
chunked_documents = text_splitter.split_documents(documents)
print("Original Documents :", len(documents))
print("Total Chunks :", len(chunked_documents))
print("\nFirst Chunk\n")
print(chunked_documents[0])

Original Documents : 50
Total Chunks : 50

First Chunk

page_content='Song Name : Believer
Artist : Imagine Dragons
Genre : Rock
Mood : Energetic
Language : English
Release Year : 2017
Keywords : motivation, power, workout, energy
Description : A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.' metadata={'id': 1, 'song_name': 'Believer', 'artist': 'Imagine Dragons', 'genre': 'Rock', 'mood': 'Energetic', 'tempo': 'Fast', 'language': 'English', 'release_year': 2017, 'keywords': ['motivation', 'power', 'workout', 'energy'], 'description': 'A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.'}


In [28]:
# import os
# db_path = "chroma_db"
# if os.path.exists(db_path):
#     print("Loading Existing Chroma Database...")
#     vector_db = Chroma(
#         persist_directory=db_path,
#         embedding_function=embedding_model
#     )

# else:
#     print("Creating New Chroma Database...")
#     vector_db = Chroma.from_documents(
#         documents=chunked_documents,
#         embedding=embedding_model,
#         persist_directory=db_path
#     )
# print("\nVector Database Ready!")
# print("Total Chunks Stored :", vector_db._collection.count())

In [32]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully!


In [33]:
vector_db = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embedding_model
)
print("Total Chunks Stored :", vector_db._collection.count())

Chroma Vector Database Created Successfully!
Total Chunks Stored : 50


In [40]:
def get_user_query():
    query = input("Enter your song preference: ").strip()
    return query


In [44]:
# query = input("enter the query")
results = vector_db.similarity_search_with_score(
    query=query,
    k=5          # Return the top 5 most similar songs
)

print(f"Query : {query}\n")

# Display Results
for i,(document,score)in enumerate(results, start=1):
    print(f"Recommendation {i}")
    print(f"Similarity Score : {score:.4f}")
    print("\nMetadata:")
    print(document.metadata)
    print("\nContent:")
    print(document.page_content)
    print("\n")


Query : energetic songs

Recommendation 1
Similarity Score : 0.8361

Metadata:
{'genre': 'Dance', 'keywords': ['dance', 'party', 'energy'], 'id': 22, 'artist': 'Ved Sharma', 'description': 'A high-energy Bollywood dance song with powerful electronic beats.', 'mood': 'Energetic', 'language': 'Hindi', 'song_name': 'Malang', 'release_year': 2020, 'tempo': 'Fast'}

Content:
Song Name : Malang
Artist : Ved Sharma
Genre : Dance
Mood : Energetic
Language : Hindi
Release Year : 2020
Keywords : dance, party, energy
Description : A high-energy Bollywood dance song with powerful electronic beats.


Recommendation 2
Similarity Score : 0.8614

Metadata:
{'release_year': 2017, 'artist': 'Imagine Dragons', 'tempo': 'Fast', 'mood': 'Energetic', 'keywords': ['motivation', 'power', 'workout', 'energy'], 'song_name': 'Believer', 'description': 'A powerful rock anthem with motivational lyrics, energetic vocals, and driving beats that inspire confidence and perseverance.', 'genre': 'Rock', 'id': 1, 'langua

In [ ]:
# Take filters from the user
genre = input("Enter Genre (Press Enter to Skip): ").strip().lower()
language = input("Enter Language (Press Enter to Skip): ").strip().lower()
mood = input("Enter Mood (Press Enter to Skip): ").strip().lower()
release_year = input("Release Year After (Press Enter to Skip): ").strip()
# Store filtered songs
filtered_results = []
for document, score in results:
    metadata = document.metadata-
    if genre and metadata["genre"].lower() != genre:
        continue
    if language and metadata["language"].lower() != language:
        continue
    if mood and metadata["mood"].lower() != mood:
        continue
    if release_year:
        if metadata["release_year"] <= int(release_year):
            continue
    filtered_results.append((document, score))

In [ ]:
def get_user_query():
    query = input("Enter your song preference: ").strip()
    return query
# query = get_user_query()
# print(query)

In [46]:
def get_filters():

    genre = input("Enter Genre (Press Enter to Skip): ")

    language = input("Enter Language (Press Enter to Skip): ")

    mood = input("Enter Mood (Press Enter to Skip): ")

    release_year = input("Enter Release Year After (Press Enter to Skip): ")

    return genre, language, mood, release_year

In [50]:
def semantic_search(query, k=10):

    results = vector_db.similarity_search_with_score(
        query=query,
        k=k
    )
    return results

In [51]:
def metadata_filter(
    results,
    genre="",
    language="",
    mood="",
    release_year=""
):
    # Store all matching songs
    filtered_results = []
    # Check every semantic search result
    for document, score in results:
        metadata = document.metadata
        if genre:

            if metadata["genre"].lower() != genre.lower():
                continue
        if language:

            if metadata["language"].lower() != language.lower():
                continue
        if mood:

            if metadata["mood"].lower() != mood.lower():
                continue

        if release_year:

            if metadata["release_year"] <= int(release_year):
                continue
        filtered_results.append((document, score))
    return filtered_results

In [52]:
def display_results(filtered_results):

    # Check if any songs were found
    if len(filtered_results) == 0:
        print("\nNo matching songs found.")
        return
    print("\nRecommended Songs\n")
    for i, (document, score) in enumerate(filtered_results, start=1):

        print("=" * 60)
        print(f"Recommendation {i}")
        print("=" * 60)

        print(f" Score : {score:.4f}")

        print("\nSong Details:")
        # print(document.metadata)
        print(f"Song Name     : {document.metadata['song_name']}")
        print(f"Artist        : {document.metadata['artist']}")
        print(f"Genre         : {document.metadata['genre']}")
        print(f"Language      : {document.metadata['language']}")
        print(f"Mood          : {document.metadata['mood']}")
        print(f"Release Year  : {document.metadata['release_year']}")

        print("\nDescription:")
        print(document.page_content)

        print("\n")

In [53]:
# ==========================================================
# FUNCTION 6 : Remove Duplicate Songs
# ==========================================================

def remove_duplicate_songs(filtered_results):

    unique_songs = []
    seen_songs = set()

    for document, score in filtered_results:

        song_name = document.metadata["song_name"]

        if song_name not in seen_songs:

            seen_songs.add(song_name)
            unique_songs.append((document, score))

    return unique_songs

In [54]:

query = get_user_query()

genre, language, mood, release_year = get_filters()

results = semantic_search(query)

filtered_results = metadata_filter(
    results,
    genre,
    language,
    mood,
    release_year
)
# Remove duplicate songs
unique_results = remove_duplicate_songs(filtered_results)

display_results(filtered_results)

Enter your song preference: energetic
Enter Genre (Press Enter to Skip): pop
Enter Language (Press Enter to Skip): english
Enter Mood (Press Enter to Skip): 
Enter Release Year After (Press Enter to Skip): 

Recommended Songs

Recommendation 1
 Score : 1.4952

Song Details:
Song Name     : Cheap Thrills
Artist        : Sia
Genre         : Pop
Language      : English
Mood          : Party
Release Year  : 2016

Description:
Song Name : Cheap Thrills
Artist : Sia
Genre : Pop
Mood : Party
Language : English
Release Year : 2016
Keywords : dance, party, fun
Description : An upbeat pop song about enjoying life without spending much.


Recommendation 2
 Score : 1.5472

Song Details:
Song Name     : Levitating
Artist        : Dua Lipa
Genre         : Pop
Language      : English
Mood          : Party
Release Year  : 2020

Description:
Song Name : Levitating
Artist : Dua Lipa
Genre : Pop
Mood : Party
Language : English
Release Year : 2020
Keywords : dance, party, fun
Description : A vibrant disco

In [56]:
def display_results(filtered_results):

    output = ""

    # Check if any songs were found
    if len(filtered_results) == 0:
        return "No matching songs found."

    output += "Recommended Songs\n\n"

    # Display every recommendation
    for i, (document, score) in enumerate(filtered_results, start=1):

        output += "=" * 60 + "\n"
        output += f"Recommendation {i}\n"
        output += "=" * 60 + "\n"

        output += f"Distance Score : {score:.4f}\n\n"

        output += f"Song Name     : {document.metadata['song_name']}\n"
        output += f"Artist        : {document.metadata['artist']}\n"
        output += f"Genre         : {document.metadata['genre']}\n"
        output += f"Language      : {document.metadata['language']}\n"
        output += f"Mood          : {document.metadata['mood']}\n"
        output += f"Release Year  : {document.metadata['release_year']}\n\n"

        output += "Description:\n"
        output += document.page_content + "\n\n"

    return output

In [57]:
def recommend_songs(query, genre, language, mood, release_year):

    # Perform Semantic Search
    results = semantic_search(query)

    # Apply Metadata Filters
    filtered_results = metadata_filter(
        results,
        genre,
        language,
        mood,
        release_year
    )

    # Remove Duplicate Songs
    unique_results = remove_duplicate_songs(filtered_results)

    # Return Formatted Results
    return display_results(unique_results)

In [58]:

def create_interface():

    interface = gr.Interface(

        fn = recommend_songs,

        inputs = [

            gr.Textbox(label="Search Query"),

            gr.Textbox(label="Genre"),

            gr.Textbox(label="Language"),

            gr.Textbox(label="Mood"),

            gr.Textbox(label="Release Year")

        ],

        outputs = gr.Textbox(label="Recommendations"),

        title = "Song Recommendation System",

        description = "Find songs using semantic search and metadata filtering."

    )

    return interface

In [59]:
interface = create_interface()

interface.launch(debug=True,share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21c96a6f1dc7946e66.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://21c96a6f1dc7946e66.gradio.live
